# M4 — Career-Skill Semantic Matcher (Sentence-Transformers)
Target: **Precision@5 >= 0.75**. See `docs/phases/MENTORA_Phase4_Model_Training.md` §3.

**Adapted from the plan doc's hypothetical schema to the real Phase 3 output:**
- `training_pairs.csv` actually has columns `job_title, required_skills, label`
  (not `skill_text, career_text, label` as the doc assumed) — this notebook
  uses `required_skills` as the skill-profile text and `job_title` as the
  career text.
- The doc's evaluator needs a student-centric validation set (skill profile
  -> known-correct careers) that Phase 3 didn't build (training_pairs.csv is
  job-posting-centric, not student-centric). The Precision@5 evaluator below
  is adapted to what the data actually supports: held-out positive pairs,
  asking whether the model's top-5 nearest job titles (from the full set of
  unique titles) include the true title for that row's skills.

## Setup — mount Drive, confirm GPU, install deps
Same first 3 cells in every Mentora training notebook. See `docs/phases/MENTORA_Phase4_Model_Training.md` §0.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
# Expect to see "Tesla T4" — if this errors or shows no GPU, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, then re-run.

Mounted at /content/drive
Mounted at /content/drive
Thu Oct  8 04:57:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/

In [2]:
import os
DATA = '/content/drive/MyDrive/mentora_data'
MODELS = '/content/drive/MyDrive/mentora_models'
for m in ['model1_gap_classifier', 'model2_question_generator',
          'model3_trajectory_predictor', 'model4_career_matcher', 'model5_cv_ner']:
    os.makedirs(f'{MODELS}/{m}', exist_ok=True)

In [3]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q sentence-transformers

## Load base model and training pairs

In [4]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import pandas as pd

model = SentenceTransformer('all-mpnet-base-v2')  # downloads once, ~420MB
pairs = pd.read_csv(f'{DATA}/processed/m4/training_pairs.csv')
print(pairs.shape, pairs['label'].value_counts().to_dict())

# Held out 10% of rows for the Precision@5 evaluator before building the
# training DataLoader, so the evaluator never sees its own training examples.
pairs = pairs.sample(frac=1.0, random_state=42).reset_index(drop=True)
n_val = int(0.1 * len(pairs))
val_pairs, train_pairs = pairs.iloc[:n_val], pairs.iloc[n_val:]

train_examples = [
    InputExample(texts=[row['required_skills'], row['job_title']], label=float(row['label']))
    for _, row in train_pairs.iterrows()
]
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(model)

/tmp/ipykernel_488/3704474506.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses
/tmp/ipykernel_488/3704474506.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

... (18 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(2134, 3) {0.0: 1067, 1.0: 1067}
(2134, 3) {0.0: 1067, 1.0: 1067}


## Precision@5 evaluator (adapted to the real, job-posting-centric data)

In [5]:
from sentence_transformers.evaluation import SentenceEvaluator
import numpy as np

class PrecisionAtKEvaluator(SentenceEvaluator):
    """For each held-out positive pair, treat `required_skills` as the query
    and ask: is the true `job_title` among the model's top-K nearest titles
    out of every unique title in the full training_pairs catalogue?"""
    def __init__(self, val_df, all_titles, k=5):
        self.skill_texts = val_df[val_df['label'] == 1.0]['required_skills'].tolist()
        self.true_titles = val_df[val_df['label'] == 1.0]['job_title'].tolist()
        self.all_titles = all_titles
        self.k = k

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        if not self.skill_texts:
            return 0.0
        skill_emb = model.encode(self.skill_texts, convert_to_numpy=True, show_progress_bar=False)
        title_emb = model.encode(self.all_titles, convert_to_numpy=True, show_progress_bar=False)
        title_norms = np.linalg.norm(title_emb, axis=1) + 1e-9
        hits = 0
        for s_emb, true_title in zip(skill_emb, self.true_titles):
            sims = title_emb @ s_emb / (title_norms * (np.linalg.norm(s_emb) + 1e-9))
            top_k_idx = np.argsort(-sims)[:self.k]
            top_k_titles = {self.all_titles[i] for i in top_k_idx}
            if true_title in top_k_titles:
                hits += 1
        score = hits / len(self.skill_texts)
        print(f"[epoch {epoch}] Precision@{self.k} = {score:.3f} ({hits}/{len(self.skill_texts)})")
        return score

all_titles = sorted(pairs['job_title'].unique().tolist())
evaluator = PrecisionAtKEvaluator(val_pairs, all_titles, k=5)
print(f"{len(all_titles)} unique job titles in the evaluator catalogue")

218 unique job titles in the evaluator catalogue


/tmp/ipykernel_488/1977439323.py:1: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import SentenceEvaluator


## Fine-tune
`save_best_model=True` + `output_path` pointing at Drive is this model's version of checkpoint-every-epoch — if the session disconnects, just re-run this cell (4 epochs on ~2000 pairs is fast enough to redo from scratch).

In [6]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=4,
    evaluator=evaluator,
    evaluation_steps=100,
    output_path=f'{MODELS}/model4_career_matcher',
    save_best_model=True,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Evaluator
100,No log,No log,0.557692
121,No log,No log,0.576923
200,No log,No log,0.625000
242,No log,No log,0.721154
300,No log,No log,0.730769
363,No log,No log,0.711538
400,No log,No log,0.721154
484,No log,No log,0.778846


[epoch 0.8264462809917356] Precision@5 = 0.558 (58/104)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[epoch 1.0] Precision@5 = 0.577 (60/104)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (5 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[epoch 3.0] Precision@5 = 0.712 (74/104)
[epoch 3.3057851239669422] Precision@5 = 0.721 (75/104)
[epoch 4.0] Precision@5 = 0.779 (81/104)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[epoch 4.0] Precision@5 = 0.779 (81/104)


## If you miss the target (Precision@5 < 0.75)
- Check `training_pairs.csv` has enough *hard negatives* (similar-sounding non-matches) — see Phase 3 §3.3
- Try `epochs=8`